In [ ]:
import pandas as pd
import os
from datetime import datetime, timedelta, timezone
import random
from groq import Groq
import re

In [ ]:
'''
    Based on the available data for a particular stock, stimulating the news for 01-01-2024, 02-01-2024, and 03-01-2024.
    The news is copied from the closest available date before the target date.
'''

In [ ]:
def random_utc_timestamp(fixed_date):
    rand_time = timedelta(
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    dt = fixed_date + rand_time
    return dt.strftime('%Y-%m-%d %H:%M:%S') + ' UTC'

In [ ]:
#List of stock files
lis1 = ['amgn.csv','TSM.csv','cmg.csv','orcl.csv','GE.csv','cmcsa.csv','pypl.csv','ebay.csv','biib.csv',
        'qcom.csv','AMD.csv','COST.csv','crm.csv','BABA.csv','cop.csv','CVX.csv','uso.csv','nke.csv','WFC.csv',
        'mrk.csv','aal.csv','gsk.csv','QQQ.csv','pep.csv','ABBV.csv']

lis1 = [x.upper() for x in lis1]

In [ ]:
for i in lis1:
    os.chdir("C:/Users/rahul/OneDrive/7_Learning/IISC/Courses/4.2_Data_Engineering_at_Scale/Course Material/Codes/Project/Streaming/data/nasdaq_split_chunks")
    df = pd.read_csv(i)
    df = df[['Date','Stock_symbol','Url','Textrank_summary']]
    print(f"Processing file: {i}")
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by='Date', ascending=False)
    df_temp = df.head(30).copy()

    df1 = df_temp[0:10].copy()
    df2 = df_temp[10:20].copy()
    df3 = df_temp[20:30].copy()

    fixed_date1 = datetime(2024, 1, 1)
    df1['Date'] = [random_utc_timestamp(fixed_date1) for _ in range(len(df1))]

    fixed_date1 = datetime(2024, 1, 2)
    df2['Date'] = [random_utc_timestamp(fixed_date1) for _ in range(len(df1))]

    fixed_date1 = datetime(2024, 1, 3)
    df3['Date'] = [random_utc_timestamp(fixed_date1) for _ in range(len(df1))]

    df_final = pd.concat([df1, df2, df3], ignore_index=True)
    df_final.rename(columns={'Date': 'event_time','Stock_symbol': 'stock_symbol','Url': 'url','Textrank_summary': 'summary'}, inplace=True)

    os.chdir("C:/Users/rahul/OneDrive/7_Learning/IISC/Courses/4.2_Data_Engineering_at_Scale/Course Material/Codes/Project/Streaming/data/engineered_data")
    df_final.to_csv(i, index=False)

In [ ]:
'''
    Precalculating the sentiment scores for the summaries and saving them to a CSV file.
    This is done to avoid recalculating sentiment scores during streaming.
'''

In [ ]:
grok_api_key = 'YOUR_GROQ_API' 

In [ ]:
def get_stock_sentiments(stock_ticker: str, news_summary: str):
    #print("Getting sentiment for:", stock_ticker)
    sentiment_template = '''
    Role:
    You are a financial analyst specializing in interpreting news sentiment for stocks. 
    Do NOT show your reasoning. Do NOT use "thinking" or any hidden tags.

    Task:
    Analyze the sentiment of the news summary toward the specified stock.

    Input:
    Stock Ticker: {stock_ticker}
    News Summary: {news_summary}

    Scoring Rules (integer only):
    1 - Negative: Clearly unfavorable news; likely negative impact on the stock.
    2 - Somewhat Negative: Slightly unfavorable; minor concerns.
    3 - Neutral: Balanced or unclear impact.
    4 - Somewhat Positive: Slightly favorable; minor optimism.
    5 - Positive: Clearly favorable; likely positive impact on the stock.

    Output:
    Return exactly ONE CHARACTER: a single digit 1,2,3,4 or 5 and nothing else.
    Do not include spaces, newline, punctuation, tags, labels, or any other characters.
    '''

    prompt = sentiment_template.format(stock_ticker=stock_ticker, news_summary=news_summary)
    client_groq = Groq(api_key=grok_api_key)

    response = client_groq.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    #print(response)
    resp_score = response.choices[0].message.content.strip()
    match = re.search(r"[1-5](?!\d)", resp_score)
    if match:
        ret_val = int(match.group(0))   # → "5"
    else:
        ret_val = 1  # Default to negative sentiment if parsing fails
    
    return ret_val

In [ ]:
input_dir = 'C:/Users/rahul/OneDrive/7_Learning/IISC/Courses/4.2_Data_Engineering_at_Scale/Course Material/Codes/Project/Streaming/data/engineered_data'
output_dir = 'C:/Users/rahul/OneDrive/7_Learning/IISC/Courses/4.2_Data_Engineering_at_Scale/Course Material/Codes/Project/Streaming/data/engineered_data_enhanced'


In [ ]:
for filename in os.listdir(input_dir):
    input_path = os.path.join(input_dir, filename)
    output_path = os.path.join(output_dir, filename)

    input_df = pd.read_csv(input_path)
    row_lis = []
    for idx in range(len(input_df)):
        sentiment = get_stock_sentiments(input_df.stock_symbol[idx], input_df.summary[idx])
        print(f"Processed file {filename}, row {idx+1}/{len(input_df)}: Sentiment={sentiment}")
        lis1 = [input_df.event_time[idx], input_df.stock_symbol[idx], input_df.url[idx], input_df.summary[idx], sentiment]
        row_lis.append(lis1)

    df_final = pd.DataFrame(row_lis, columns=['event_time', 'stock_symbol', 'url', 'summary', 'sentiment_score'])
    df_final.to_csv(output_path, index=False)
    print(f"Saved enhanced file: {output_path}")